<a href="https://colab.research.google.com/github/khalid-saqr/picoNewton/blob/main/picoNewton_v4/notebooks/picoNewton_v4_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# picoNewton_v4 — Production Model Solution
This is the single Colab entry point for the six-artery anisotropic Womersley → membrane → Piezo1 calculation. Run all cells once from top to bottom. It creates a timestamped Google Drive run, installs the pinned package, solves the publication-resolution model, executes the hypothesis/control matrix, generates figures and tables, and archives provenance and checksums.

**No quick mode, dry run, reduced-resolution simulation, or pytest stage is executed.**


## Scientific question
Does anisotropic near-wall forcing provide mechanosensory information distinct from wall shear stress across Aortic Root, Thoracic Aorta, Femoral, Carotid, Iliac, and Brachial arteries?

The workflow keeps wall shear stress, signed transverse Lamb force, and nonnegative Lamb-force exposure physically separate. Signed force is directional; exposure is never treated as a signed traction.


## Vector-resolved membrane and Piezo1 model
Normal and tangential mechanics are transferred through separate passive viscoelastic branches to apical and junctional membrane tension. The four-state Piezo1 model reports open probability, current, and a reduced-order calcium-scale endpoint. Predictions remain conditional on the packaged inputs, constitutive assumptions, transfer parameters, and calibration.


## Fixed production resolution
The calculation is fixed at radial order 150, 2,048 time points per cardiac cycle, 256 near-wall integration points, six arteries, the complete control matrix, and the configured sensitivity analysis. The package API names this resolution `full`; it is not a user-selectable mode here.


In [48]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [49]:
REPO_URL='https://github.com/khalid-saqr/picoNewton.git'
REPO_REF='8fd376075e90a739ca74f1afede8a895344b3054'
PACKAGE_SUBDIR='picoNewton_v4'
CALIBRATION_RELATIVE_PATH='configs/literature_calibration.json'
NUMERICAL_PROFILE='full'
RUN_SENSITIVITY_SCAN=True
MINIMUM_PASSING_ARTERIES=4
EXPECTED_ARTERIES={'Aortic Root','Thoracic Aorta','Femoral','Carotid','Iliac','Brachial'}


## Timestamped runtime
Each execution writes to `MyDrive/picoNewton_v4_runtime/runs/YYYYMMDD_HHMMSS_UTC_<suffix>/`. Colab local storage is used for computation, then the complete result is copied to Drive. Existing runs are never overwritten.


In [50]:
from __future__ import annotations
import hashlib,json,os,platform,secrets,shutil,subprocess,sys
from datetime import datetime,timezone
from pathlib import Path
for key in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','NUMEXPR_NUM_THREADS'): os.environ[key]='2'
RUN_ID=f"{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_UTC')}_{secrets.token_hex(2)}"
DRIVE_ROOT=Path('/content/drive/MyDrive/picoNewton_v4_runtime')
RUN_DIR=DRIVE_ROOT/'runs'/RUN_ID
LOCAL_ROOT=Path('/content/picoNewton_v4_work')/RUN_ID
REPO_DIR=LOCAL_ROOT/'repository'
PACKAGE_DIR=REPO_DIR/PACKAGE_SUBDIR
LOCAL_OUTPUT=LOCAL_ROOT/'model_outputs'
for p in (RUN_DIR/'inputs',RUN_DIR/'configs',RUN_DIR/'logs',RUN_DIR/'provenance',RUN_DIR/'environment'): p.mkdir(parents=True,exist_ok=True)
print('Run ID:',RUN_ID); print('Drive output:',RUN_DIR)


Run ID: 20260719_043211_UTC_c9da
Drive output: /content/drive/MyDrive/picoNewton_v4_runtime/runs/20260719_043211_UTC_c9da


## Install the pinned package
The exact Git commit is cloned and installed without development/test dependencies. Installation failure stops execution before the model starts. No package test suite is run.


In [55]:
def run_logged(cmd,log,cwd='/content'):
    cmd=[str(x) for x in cmd]
    # Ensure the parent directory for the log exists
    Path(log).parent.mkdir(parents=True, exist_ok=True)
    r=subprocess.run(cmd,cwd=cwd,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
    Path(log).write_text('$ '+' '.join(cmd)+'\n\n'+r.stdout,encoding='utf-8')
    print(r.stdout[-3000:])
    if r.returncode: raise RuntimeError(f'Command failed ({r.returncode}): {cmd}')

if REPO_DIR.exists(): shutil.rmtree(REPO_DIR)
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

# Run git clone with an explicit, stable cwd
run_logged(['git','clone','--filter=blob:none','--no-checkout',REPO_URL,REPO_DIR], RUN_DIR/'logs'/'git_clone.log', cwd='/content')
run_logged(['git','-C',REPO_DIR,'fetch','--depth','1','origin',REPO_REF], RUN_DIR/'logs'/'git_fetch.log', cwd='/content')
run_logged(['git','-C',REPO_DIR,'checkout','--detach','FETCH_HEAD'], RUN_DIR/'logs'/'git_checkout.log', cwd='/content')

GIT_SHA=subprocess.check_output(['git','-C',REPO_DIR,'rev-parse','HEAD'],text=True).strip()
if GIT_SHA!=REPO_REF: raise RuntimeError(f'Commit mismatch: {GIT_SHA}')
if not PACKAGE_DIR.is_dir(): raise FileNotFoundError(PACKAGE_DIR)
run_logged([sys.executable,'-m','pip','install','-e',PACKAGE_DIR], RUN_DIR/'logs'/'pip_install.log', cwd='/content')
(RUN_DIR/'provenance'/'git_commit.txt').write_text(GIT_SHA+'\n',encoding='utf-8')

Cloning into '/content/picoNewton_v4_work/20260719_043211_UTC_c9da/repository'...

From https://github.com/khalid-saqr/picoNewton
 * branch            8fd376075e90a739ca74f1afede8a895344b3054 -> FETCH_HEAD

HEAD is now at 8fd3760 Add files via upload

/python3.12/dist-packages (from piconewton-v4==4.0.0) (2.0.2)
  Building editable for piconewton-v4 (pyproject.toml): started
  Building editable for piconewton-v4 (pyproject.toml): finished with status 'done'
  Created wheel for piconewton-v4: filename=piconewton_v4-4.0.0-0.editable-py3-none-any.whl size=3726 sha256=5458ac5c895613ab2ad4b57500344cf4c43ac27bc3c49627be277e3785ab1598
  Stored in directory: /tmp/pip-ephem-wheel-cache-9iuranzc/wheels/59/da/ec/bedc7d7309bdda9ec969d8f21e652cac1e38637122146051bf
Successfully built piconewton-v4
  Attempting uninstall: piconewton-v4
    Found existing installation: piconewton-v4 4.0.0
    Uninstalling piconewton-v4-4.0.0:
      Successfully uninstalled piconewton-v4-4.0.0



41

## Freeze scientific inputs
Before solving, the notebook confirms that the model configuration, calibration, and artery table exist and that the table contains exactly the required six arteries. These are input-integrity checks, not a preliminary simulation.


In [58]:
# Freeze scientific inputs and verify the active package import.

import importlib
import json
import shutil
import sys
from pathlib import Path

import pandas as pd


# ---------------------------------------------------------------------
# 1. Make the src-layout package visible to the current Colab kernel.
# ---------------------------------------------------------------------

PACKAGE_DIR = Path(PACKAGE_DIR).resolve()
SOURCE_DIR = PACKAGE_DIR / "src"
PACKAGE_INIT = SOURCE_DIR / "piconewton_v4" / "__init__.py"

if not PACKAGE_DIR.is_dir():
    raise FileNotFoundError(f"Package directory does not exist: {PACKAGE_DIR}")

if not PACKAGE_INIT.is_file():
    raise FileNotFoundError(
        "The picoNewton_v4 src-layout package was not found.\n"
        f"Expected file: {PACKAGE_INIT}"
    )

source_path = str(SOURCE_DIR)

# Put the source tree first so an old or incomplete installation cannot win.
if source_path in sys.path:
    sys.path.remove(source_path)
sys.path.insert(0, source_path)

# Remove modules cached by an earlier failed notebook execution.
for module_name in list(sys.modules):
    if module_name == "piconewton_v4" or module_name.startswith("piconewton_v4."):
        del sys.modules[module_name]

importlib.invalidate_caches()

piconewton_v4 = importlib.import_module("piconewton_v4")
load_parameterization = importlib.import_module(
    "piconewton_v4.calibration"
).load_parameterization

module_path = Path(piconewton_v4.__file__).resolve()

if SOURCE_DIR not in module_path.parents:
    raise RuntimeError(
        "piconewton_v4 was imported from the wrong location.\n"
        f"Expected source directory: {SOURCE_DIR}\n"
        f"Actual module path: {module_path}"
    )

if piconewton_v4.__version__ != "4.0.0":
    raise RuntimeError(
        f"Expected piconewton_v4 version 4.0.0, "
        f"but imported {piconewton_v4.__version__}"
    )

print(f"Imported piconewton_v4 {piconewton_v4.__version__}")
print(f"Import path: {module_path}")


# ---------------------------------------------------------------------
# 2. Locate and validate the scientific inputs.
# ---------------------------------------------------------------------

CALIBRATION_PATH = PACKAGE_DIR / CALIBRATION_RELATIVE_PATH
MODEL_CONFIG_PATH = PACKAGE_DIR / "configs" / "model.json"
ARTERY_INPUT_PATH = PACKAGE_DIR / "data" / "ground_truth_arteries.csv"

required_files = (
    CALIBRATION_PATH,
    MODEL_CONFIG_PATH,
    ARTERY_INPUT_PATH,
)

missing_files = [path for path in required_files if not path.is_file()]

if missing_files:
    raise FileNotFoundError(
        "Required scientific input files are missing:\n"
        + "\n".join(str(path) for path in missing_files)
    )


# ---------------------------------------------------------------------
# 3. Enforce the six-artery input contract.
# ---------------------------------------------------------------------

arteries = pd.read_csv(ARTERY_INPUT_PATH)

artery_column = next(
    (
        column
        for column in arteries.columns
        if column.strip().lower() in {"artery", "artery_name", "name"}
    ),
    None,
)

if artery_column is None:
    raise ValueError(
        "No artery-name column was found in "
        f"{ARTERY_INPUT_PATH.name}. Columns: {list(arteries.columns)}"
    )

actual_arteries = set(
    arteries[artery_column]
    .astype(str)
    .str.strip()
)

if actual_arteries != EXPECTED_ARTERIES:
    raise ValueError(
        "Six-artery input contract failed.\n"
        f"Expected: {sorted(EXPECTED_ARTERIES)}\n"
        f"Found: {sorted(actual_arteries)}"
    )


# ---------------------------------------------------------------------
# 4. Load model configuration and calibration.
# ---------------------------------------------------------------------

model_configuration = json.loads(
    MODEL_CONFIG_PATH.read_text(encoding="utf-8")
)

(
    interface_parameters,
    endpoint_parameters,
    calibration_audit,
) = load_parameterization(CALIBRATION_PATH)


# ---------------------------------------------------------------------
# 5. Copy immutable inputs to the persistent run directory.
# ---------------------------------------------------------------------

input_destinations = (
    (
        ARTERY_INPUT_PATH,
        RUN_DIR / "inputs" / ARTERY_INPUT_PATH.name,
    ),
    (
        MODEL_CONFIG_PATH,
        RUN_DIR / "configs" / MODEL_CONFIG_PATH.name,
    ),
    (
        CALIBRATION_PATH,
        RUN_DIR / "configs" / CALIBRATION_PATH.name,
    ),
)

for source, destination in input_destinations:
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)


# ---------------------------------------------------------------------
# 6. Record provenance.
# ---------------------------------------------------------------------

input_record = {
    "run_id": RUN_ID,
    "git_sha": GIT_SHA,
    "package_version": piconewton_v4.__version__,
    "package_import_path": str(module_path),
    "profile": NUMERICAL_PROFILE,
    "sensitivity_scan": RUN_SENSITIVITY_SCAN,
    "arteries": sorted(EXPECTED_ARTERIES),
    "calibration_status": calibration_audit.get("calibration_status"),
    "calibration_complete": calibration_audit.get("complete"),
    "missing_source_groups": calibration_audit.get(
        "missing_source_groups",
        [],
    ),
}

input_record_path = RUN_DIR / "provenance" / "input_record.json"
input_record_path.parent.mkdir(parents=True, exist_ok=True)
input_record_path.write_text(
    json.dumps(
        input_record,
        indent=2,
        sort_keys=True,
        default=str,
    )
    + "\n",
    encoding="utf-8",
)

print(json.dumps(input_record, indent=2, sort_keys=True, default=str))
print("Scientific inputs frozen successfully.")

Imported piconewton_v4 4.0.0
Import path: /content/picoNewton_v4_work/20260719_043211_UTC_c9da/repository/picoNewton_v4/src/piconewton_v4/__init__.py
{
  "arteries": [
    "Aortic Root",
    "Brachial",
    "Carotid",
    "Femoral",
    "Iliac",
    "Thoracic Aorta"
  ],
  "calibration_complete": false,
  "calibration_status": "literature_constrained_independent_evidence",
  "git_sha": "8fd376075e90a739ca74f1afede8a895344b3054",
  "missing_source_groups": [],
  "package_import_path": "/content/picoNewton_v4_work/20260719_043211_UTC_c9da/repository/picoNewton_v4/src/piconewton_v4/__init__.py",
  "package_version": "4.0.0",
  "profile": "full",
  "run_id": "20260719_043211_UTC_c9da",
  "sensitivity_scan": true
}
Scientific inputs frozen successfully.


## Solve the production model
This is the only numerical solution cell. It runs all six arteries at the fixed publication resolution and executes the configured sensitivity analysis.


In [62]:
import json

manifest_path = LOCAL_OUTPUT / "manifest.json"

if manifest_path.is_file():
    print("Using the production result already computed in this session.")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
else:
    from piconewton_v4.workflow import run_workflow

    manifest = run_workflow(
        package_root=PACKAGE_DIR,
        output_root=LOCAL_OUTPUT,
        profile=NUMERICAL_PROFILE,
        calibration_path=CALIBRATION_PATH,
        run_scan=RUN_SENSITIVITY_SCAN,
    )

EXPECTED_STATUS = "passed_structural_validation"

if manifest.get("status") != EXPECTED_STATUS:
    raise RuntimeError(
        f"Workflow returned {manifest.get('status')!r}; "
        f"expected {EXPECTED_STATUS!r}."
    )

print("Workflow status:", manifest["status"])
print("Completed UTC:", manifest.get("completed_utc"))
print("Production outputs:", LOCAL_OUTPUT)

Using the production result already computed in this session.
Workflow status: passed_structural_validation
Completed UTC: 2026-07-19T04:37:07.874814+00:00
Production outputs: /content/picoNewton_v4_work/20260719_043211_UTC_c9da/model_outputs


## Validate and classify the completed result
The completed outputs are checked for six-artery completeness, probability conservation, passive mechanics, and correct separation of signed force from exposure. Detection thresholds are then applied and the standard figures are generated.


In [63]:
from piconewton_v4.hypotheses import DecisionThresholds,write_decisions
from piconewton_v4.reporting import generate_standard_figures
from piconewton_v4.validation import validate_output_directory
validation_report=validate_output_directory(LOCAL_OUTPUT)
if not validation_report.get('passed'): raise RuntimeError(validation_report)
thresholds=DecisionThresholds(current_rms_pa=endpoint_parameters.current_detection_limit_pa,calcium_rms_nm=endpoint_parameters.calcium_detection_limit_nm,minimum_arteries=MINIMUM_PASSING_ARTERIES)
decisions=write_decisions(LOCAL_OUTPUT/'hypothesis_effects.csv',LOCAL_OUTPUT/'hypothesis_decisions.csv',thresholds,LOCAL_OUTPUT/'decision_thresholds.json')
figure_paths=generate_standard_figures(LOCAL_OUTPUT,LOCAL_OUTPUT/'figures')
(LOCAL_OUTPUT/'output_validation.json').write_text(json.dumps(validation_report,indent=2)+'\n')
print(json.dumps(validation_report,indent=2)); print('Figures:',len(figure_paths))


{
  "passed": true,
  "arteries": 6,
  "summary_rows": 30,
  "effect_rows": 120,
  "waveform_arrays": 576,
  "maximum_probability_sum_error": 2.2726265314076954e-13,
  "minimum_probability": 3.0898517890857613e-07
}
Figures: 8


## Principal tables
The artery summary, effect sizes, and hypothesis decisions are displayed below. Complete arrays and all generated files are archived.


In [65]:
from IPython.display import display
import pandas as pd

TABLES = {
    "Six-artery summary": LOCAL_OUTPUT / "six_artery_summary.csv",
    "Hypothesis effects": LOCAL_OUTPUT / "hypothesis_effects.csv",
    "Hypothesis decisions": LOCAL_OUTPUT / "hypothesis_decisions.csv",
}

missing = [path for path in TABLES.values() if not path.is_file()]

if missing:
    available = sorted(
        str(path.relative_to(LOCAL_OUTPUT))
        for path in LOCAL_OUTPUT.rglob("*")
        if path.is_file()
    )

    raise FileNotFoundError(
        "Required principal table files are missing:\n"
        + "\n".join(f"  - {path}" for path in missing)
        + "\n\nFiles currently available in LOCAL_OUTPUT:\n"
        + (
            "\n".join(f"  - {name}" for name in available)
            if available
            else "  <none>"
        )
    )

principal_tables = {}

for title, path in TABLES.items():
    frame = pd.read_csv(path)
    principal_tables[title] = frame

    print(f"\n{title}")
    print(f"File: {path.name}")
    print(f"Rows: {len(frame):,} | Columns: {len(frame.columns):,}")
    display(frame)

print("\nAll principal tables loaded successfully.")


Six-artery summary
File: six_artery_summary.csv
Rows: 30 | Columns: 27


,artery_id,pathway,popen_mean,popen_dynamic_range,current_mean_abs_pa,current_rms_pa,current_peak_abs_pa,current_dynamic_range_pa,charge_abs_fc_per_cycle,calcium_mean_nm,...,junctional_calcium_peak_nm,peak_time_fraction,current_h1_amplitude_pa,current_h3_6_power_fraction,apical_pressure_peak_mmhg,junctional_pressure_peak_mmhg,force_angle_range_rad,probability_sum_error,minimum_probability,exposure_used_as_signed_load
0,aortic_root,zero,0.002015,3.165870e-17,10.071182,10.071182,10.071182,1.598721e-13,8392.651942,19332.397913,...,19332.397913,0.028809,4.257468e-15,0.163970,9.375938,9.375938,0.000000,2.272627e-13,2.015042e-03,False
1,aortic_root,wss,0.001463,5.921021e-04,7.309916,7.368061,8.470795,2.959326e+00,6091.596392,14031.937246,...,8752.560327,0.706055,9.395669e-01,0.058638,9.375938,70.000000,3.141593,2.272627e-13,4.618561e-07,False
2,aortic_root,signed,0.001820,1.609971e-03,9.095064,9.327438,15.770378,8.046635e+00,7579.219821,17458.664373,...,15643.658970,0.061523,2.001867e+00,0.290427,9.375938,27.237250,0.000000,2.272627e-13,1.075691e-03,False
3,aortic_root,exposure,0.001820,1.609971e-03,9.095064,9.327438,15.770378,8.046635e+00,7579.219809,17458.664346,...,19332.397913,0.061523,2.001867e+00,0.290427,27.237250,9.375938,0.000000,2.272627e-13,1.075691e-03,False
4,aortic_root,vector,0.001175,1.901299e-03,5.874792,6.289488,12.775928,9.502694e+00,4895.660249,11277.109147,...,6980.026713,0.061035,2.091578e+00,0.267660,27.237250,70.000000,3.141441,1.902922e-13,3.089852e-07,False
5,thoracic_aorta,zero,0.002015,3.165870e-17,10.071182,10.071182,10.071182,1.598721e-13,8392.651942,19332.397913,...,19332.397913,0.028809,4.257468e-15,0.163970,9.375938,9.375938,0.000000,2.272627e-13,2.015042e-03,False
6,thoracic_aorta,wss,0.001549,7.374389e-04,7.742488,7.837824,9.263783,3.685720e+00,6452.073177,14862.292262,...,10424.170128,0.733887,1.220241e+00,0.075931,9.375938,55.060647,3.141593,2.272627e-13,8.345451e-06,False
7,thoracic_aorta,signed,0.001901,9.803184e-04,9.502015,9.587087,13.469378,4.899631e+00,7918.345444,18239.837181,...,17185.407915,0.070801,1.237310e+00,0.280897,9.375938,18.930670,0.000000,2.272627e-13,1.414228e-03,False
8,thoracic_aorta,exposure,0.001901,9.803184e-04,9.502015,9.587087,13.469378,4.899631e+00,7918.345439,18239.837169,...,19332.397913,0.070801,1.237310e+00,0.280897,18.930670,9.375938,0.000000,2.272627e-13,1.414228e-03,False
9,thoracic_aorta,vector,0.001364,1.459560e-03,6.818086,7.059633,11.429833,7.294880e+00,5681.738721,13087.833805,...,9049.709685,0.068848,1.752083e+00,0.210697,18.930670,64.418985,3.141473,5.773160e-15,1.318555e-06,False



Hypothesis effects
File: hypothesis_effects.csv
Rows: 120 | Columns: 10


,artery_id,hypothesis,comparator,target,current_rms_difference_pa,current_peak_difference_pa,current_relative_rms,calcium_rms_difference_nm,calcium_peak_difference_nm,calcium_relative_rms
0,aortic_root,activation,zero,signed,2.287733,5.699195,0.284309,1873.834934,1903.272684,31.812101
1,aortic_root,activation,zero,exposure,2.287733,5.699195,0.284309,1873.834961,1903.272711,31.812102
2,aortic_root,H3a,wss,signed,2.669589,7.887367,0.331765,3426.781581,3445.977502,58.176481
3,aortic_root,H3a,wss_abs,exposure,3.753959,9.827319,0.466525,5979.704543,6009.238342,101.517462
4,aortic_root,H3a_rms_matched,wss,signed_rms_matched,0.811961,1.671495,0.666198,239.482765,249.451361,23.277536
...,...,...,...,...,...,...,...,...,...,...
115,carotid,H3b,wss_exposure_surrogate,actual_exposure,0.351065,0.981349,0.122276,133.606319,137.621775,5.793235
116,iliac,H3b,wss_signed_surrogate,actual_signed,0.566549,1.231913,0.395800,427.934968,437.640258,35.111804
117,iliac,H3b,wss_exposure_surrogate,actual_exposure,0.243613,0.622618,0.170192,166.063991,169.268333,13.625449
118,brachial,H3b,wss_signed_surrogate,actual_signed,0.533020,0.940683,0.711421,769.867876,776.670725,119.849970



Hypothesis decisions
File: hypothesis_decisions.csv
Rows: 20 | Columns: 7


,hypothesis,target,passing_arteries,required_arteries,decision,current_threshold_pa,calcium_threshold_nm
0,H3a,exposure,6,4,pass,3.3,10.0
1,H3a,signed,6,4,pass,3.3,10.0
2,H3a_peak_matched,exposure_peak_matched,6,4,pass,3.3,10.0
3,H3a_peak_matched,signed_peak_matched,6,4,pass,3.3,10.0
4,H3a_rms_matched,exposure_rms_matched,6,4,pass,3.3,10.0
5,H3a_rms_matched,signed_rms_matched,6,4,pass,3.3,10.0
6,H3a_work_matched,exposure_work_matched,6,4,pass,3.3,10.0
7,H3a_work_matched,signed_work_matched,6,4,pass,3.3,10.0
8,H3b,actual_exposure,6,4,pass,3.3,10.0
9,H3b,actual_signed,6,4,pass,3.3,10.0



All principal tables loaded successfully.


## Archive and checksum
The complete native output tree, Python environment, run manifest, and SHA-256 checksums are stored in the timestamped Google Drive directory.


In [67]:
# Archive the completed run and generate reproducibility checksums.

from __future__ import annotations

import hashlib
import importlib.metadata
import json
import platform
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path


LOCAL_OUTPUT = Path(LOCAL_OUTPUT)
RUN_DIR = Path(RUN_DIR)

if not LOCAL_OUTPUT.is_dir():
    raise FileNotFoundError(
        f"Completed model-output directory not found: {LOCAL_OUTPUT}"
    )

required_outputs = (
    LOCAL_OUTPUT / "manifest.json",
    LOCAL_OUTPUT / "validation.json",
    LOCAL_OUTPUT / "six_artery_summary.csv",
    LOCAL_OUTPUT / "hypothesis_effects.csv",
    LOCAL_OUTPUT / "hypothesis_decisions.csv",
)

missing_outputs = [
    path for path in required_outputs
    if not path.is_file()
]

if missing_outputs:
    raise FileNotFoundError(
        "Required completed outputs are missing:\n"
        + "\n".join(f"  - {path}" for path in missing_outputs)
    )


# Recover variables if this cell is run independently.
if "manifest" not in globals():
    manifest = json.loads(
        (LOCAL_OUTPUT / "manifest.json").read_text(
            encoding="utf-8"
        )
    )

if "validation_report" not in globals():
    validation_path = LOCAL_OUTPUT / "output_validation.json"

    if not validation_path.is_file():
        validation_path = LOCAL_OUTPUT / "validation.json"

    validation_report = json.loads(
        validation_path.read_text(encoding="utf-8")
    )

if "figure_paths" not in globals():
    figures_directory = LOCAL_OUTPUT / "figures"

    figure_paths = (
        sorted(
            path
            for path in figures_directory.rglob("*")
            if path.is_file()
        )
        if figures_directory.is_dir()
        else []
    )


EXPECTED_STATUS = "passed_structural_validation"

if manifest.get("status") != EXPECTED_STATUS:
    raise RuntimeError(
        f"Refusing to archive status "
        f"{manifest.get('status')!r}; "
        f"expected {EXPECTED_STATUS!r}."
    )

if not validation_report.get("passed", False):
    raise RuntimeError(
        "Refusing to archive because output validation did not pass."
    )


# Create the persistent archive.
environment_dir = RUN_DIR / "environment"
provenance_dir = RUN_DIR / "provenance"
archived_output_dir = RUN_DIR / "model_outputs"

environment_dir.mkdir(parents=True, exist_ok=True)
provenance_dir.mkdir(parents=True, exist_ok=True)

if archived_output_dir.exists():
    shutil.rmtree(archived_output_dir)

shutil.copytree(
    LOCAL_OUTPUT,
    archived_output_dir,
)


# Generate the package inventory directly through Python.
# This does not depend on the pip executable.
installed_packages = []

for distribution in importlib.metadata.distributions():
    name = (
        distribution.metadata.get("Name")
        or "unknown-package"
    )
    version = distribution.version or "unknown-version"
    installed_packages.append(f"{name}=={version}")

installed_packages = sorted(
    set(installed_packages),
    key=str.lower,
)

(environment_dir / "python_packages.txt").write_text(
    "\n".join(installed_packages) + "\n",
    encoding="utf-8",
)


# pip freeze is supplementary. Its failure will be recorded,
# but it will not abort the scientific archive.
pip_command = [
    sys.executable,
    "-m",
    "pip",
    "freeze",
]

pip_result = subprocess.run(
    pip_command,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    check=False,
)

pip_record = (
    "$ "
    + " ".join(pip_command)
    + f"\nreturn_code={pip_result.returncode}\n\n"
    + pip_result.stdout
)

(environment_dir / "pip_freeze.txt").write_text(
    pip_record,
    encoding="utf-8",
)


package_version = getattr(
    piconewton_v4,
    "__version__",
    "unknown",
)

system_record = {
    "run_id": RUN_ID,
    "completed_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "python": sys.version,
    "python_executable": sys.executable,
    "platform": platform.platform(),
    "package_version": package_version,
    "git_sha": GIT_SHA,
    "profile": NUMERICAL_PROFILE,
    "sensitivity_scan": RUN_SENSITIVITY_SCAN,
    "workflow_status": manifest.get("status"),
    "validation_passed": bool(
        validation_report.get("passed")
    ),
    "figure_count": len(figure_paths),
    "environment_inventory": "importlib.metadata",
    "pip_freeze_return_code": pip_result.returncode,
}

(environment_dir / "system.json").write_text(
    json.dumps(
        system_record,
        indent=2,
        sort_keys=True,
        default=str,
    )
    + "\n",
    encoding="utf-8",
)

run_manifest = {
    **system_record,
    "arteries": sorted(EXPECTED_ARTERIES),
    "calibration_file": CALIBRATION_RELATIVE_PATH,
    "local_output_source": str(LOCAL_OUTPUT),
    "archived_output": str(archived_output_dir),
}

(provenance_dir / "run_manifest.json").write_text(
    json.dumps(
        run_manifest,
        indent=2,
        sort_keys=True,
        default=str,
    )
    + "\n",
    encoding="utf-8",
)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as stream:
        for block in iter(
            lambda: stream.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


checksum_path = (
    provenance_dir / "SHA256SUMS.json"
)

# Exclude the checksum file itself.
checksums = {
    str(path.relative_to(RUN_DIR)): sha256_file(path)
    for path in sorted(RUN_DIR.rglob("*"))
    if path.is_file() and path != checksum_path
}

checksum_path.write_text(
    json.dumps(
        checksums,
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)


# Verify the finished archive.
archive_required = (
    provenance_dir / "run_manifest.json",
    provenance_dir / "SHA256SUMS.json",
    environment_dir / "system.json",
    environment_dir / "python_packages.txt",
    archived_output_dir / "six_artery_summary.csv",
    archived_output_dir / "hypothesis_effects.csv",
    archived_output_dir / "hypothesis_decisions.csv",
)

archive_missing = [
    path for path in archive_required
    if not path.is_file()
]

if archive_missing:
    raise RuntimeError(
        "Archive verification failed:\n"
        + "\n".join(
            f"  - {path}"
            for path in archive_missing
        )
    )

print("Production run archived successfully.")
print("Run ID:", RUN_ID)
print("Persistent output:", RUN_DIR)
print("Workflow status:", manifest["status"])
print(
    "Validation passed:",
    validation_report["passed"],
)
print(
    "Archived files checksummed:",
    len(checksums),
)

if pip_result.returncode != 0:
    print(
        "Note: pip freeze was unavailable. "
        "The complete package inventory was recorded "
        "through importlib.metadata instead."
    )

Production run archived successfully.
Run ID: 20260719_043211_UTC_c9da
Persistent output: /content/drive/MyDrive/picoNewton_v4_runtime/runs/20260719_043211_UTC_c9da
Workflow status: passed_structural_validation
Validation passed: True
Archived files checksummed: 34
Note: pip freeze was unavailable. The complete package inventory was recorded through importlib.metadata instead.
